# Assignment 8: Streamflow prediction from rainfall with neural networks

Predicting how much water will be in a river in a few days' time is the core of flood
forecasting, reservoir operation and drought management. The physical problem is
**rainfall-runoff**: rain falls on a catchment, some evaporates, some soaks into the soil,
some runs off into the channel, and the river responds hours to days later. That response is
strongly **nonlinear** (the same 20 mm of rain produces a trickle on dry ground in August and a flood on saturated ground in March) and it depends on the *history* of the catchment,
not just today's weather.

That combination makes it a natural neural network problem, and it is one of the areas where
machine learning has most clearly outperformed traditional conceptual hydrological models.

You will use two real public datasets:

- **USGS streamflow**: daily mean discharge measured at gauging stations, the backbone of
  US water resource monitoring.
- **Daymet**: daily gridded precipitation and temperature for North America, from Oak Ridge
  National Laboratory.

Both are free and need no API key. You will build a **regional** model: one network trained
on six catchments at once, rather than a separate model per river.

Answer each numbered question in the empty cell below it.

In [ ]:
import io
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

## Load the data

Six gauged catchments, chosen to be small enough that a single Daymet grid cell is a
reasonable summary of the rain falling on them, and to span a range of climates from
northern Maine to the southern Appalachians. `drainage_sq_mi` is the catchment area.

This downloads twelve time series and takes a couple of minutes.

In [ ]:
BASINS = {
    # site_no: (latitude, longitude, drainage area in square miles, name)
    "02177000": (34.8138, -83.3064,  207, "Chattooga River, GA"),
    "01547700": (41.0595, -77.6058, 44.1, "Marsh Creek, PA"),
    "01013500": (47.2375, -68.5828,  870, "Fish River, ME"),
    "01543500": (41.3173, -78.1031,  685, "Sinnemahoning Creek, PA"),
    "02143040": (35.6640, -81.2140, 62.0, "Henry Fork, NC"),
    "03164000": (36.6473, -80.9790, 1141, "New River, VA"),
}
START, END = "1995-01-01", "2020-12-31"


def load_basin(site, lat, lon, area, name):
    """Daily discharge from USGS plus daily weather from Daymet, joined on date."""
    # USGS daily values, parameter 00060 = discharge in cubic feet per second.
    url = (f"https://waterservices.usgs.gov/nwis/dv/?format=rdb&sites={site}"
           f"&startDT={START}&endDT={END}&parameterCd=00060&siteStatus=all")
    text = urllib.request.urlopen(url).read().decode()
    rows = [line.split("\t") for line in text.splitlines() if line.startswith("USGS")]
    flow = pd.DataFrame(rows).iloc[:, [2, 3]]
    flow.columns = ["date", "discharge_cfs"]
    flow["date"] = pd.to_datetime(flow["date"])
    flow["discharge_cfs"] = pd.to_numeric(flow["discharge_cfs"], errors="coerce")

    # Daymet single-pixel extraction at the gauge location.
    url = (f"https://daymet.ornl.gov/single-pixel/api/data?lat={lat}&lon={lon}"
           f"&vars=prcp,tmax,tmin&start={START}&end={END}")
    met = pd.read_csv(io.StringIO(urllib.request.urlopen(url).read().decode()), skiprows=6)
    met.columns = ["year", "yday", "prcp", "tmax", "tmin"]
    met["date"] = (pd.to_datetime(met["year"].astype(str), format="%Y")
                   + pd.to_timedelta(met["yday"] - 1, unit="D"))

    df = met.merge(flow, on="date", how="inner").sort_values("date").reset_index(drop=True)
    df["site"] = site
    df["name"] = name
    df["area_sq_mi"] = area
    df["lat"] = lat
    df["lon"] = lon
    return df


frames = []
for site, (lat, lon, area, name) in BASINS.items():
    d = load_basin(site, lat, lon, area, name)
    print(f"{name:<26} {len(d):>6,} days  "
          f"{d.date.min().date()} to {d.date.max().date()}")
    frames.append(d)

raw = pd.concat(frames, ignore_index=True)
print(f"\ntotal: {len(raw):,} basin-days across {raw.site.nunique()} catchments")
raw.head()

```{admonition} Note on Daymet's calendar
:class: note
Daymet uses a 365-day year and omits 31 December in leap years, so the inner join above
silently drops those days. That is fine here, but it is the kind of thing worth noticing
when you join two climate datasets: a silent loss of six days per two decades is easy to
miss and not always harmless.
```

## Part 1: Explore the data

1. Plot the discharge time series for one catchment over a five-year window, with a
logarithmic y-axis. Describe the shape of the hydrograph, what do the rises look like
compared with the falls, and why are they asymmetric?

2. Plot the distribution of daily discharge for all catchments together, first on a linear
scale and then after a $\log(1+x)$ transform. Which is easier to model with a squared-error
loss, and why does streamflow have this shape?

3. Make a scatter plot of same-day discharge against same-day precipitation for one
catchment, and compute the correlation. It will be much weaker than you might expect.
Give two physical reasons why today's rainfall alone is a poor predictor of today's flow.

4. Catchments differ enormously in size, so their discharges are not comparable, the New
River drains 26 times the area of Marsh Creek. Create a column `logQ` holding
$\log(1 + Q/A)$, where $Q$ is discharge in cfs and $A$ is drainage area in square miles.
Plot the distribution of `logQ` for each catchment on shared axes and confirm that they now
overlap.

## Part 2: Build the features

A network given only today's weather cannot know whether the ground is already saturated.
The catchment's *memory* has to be supplied explicitly, as features describing the recent
past.

5. Working **within each catchment separately** (use `groupby("site")` so that features
never leak across basin boundaries), construct:

- `prcp_lag1` through `prcp_lag10`: precipitation on each of the previous ten days
- `prcp_sum3`, `prcp_sum7`, `prcp_sum14`, `prcp_sum30`, `prcp_sum90`: rolling
  precipitation totals, which act as proxies for how wet the catchment already is
- `tmean_7` and `tmean_30`, rolling mean temperature, a proxy for evaporative demand and
  for whether precipitation is falling as snow
- `doy_sin` and `doy_cos`, the day of year encoded as a pair of sinusoids

6. Also within each catchment, construct `logQ_lag0` through `logQ_lag3`, the
area-normalised log discharge today and on the three previous days. These are the single
most informative features you will build: rivers have long memories.

Then create the **target**, `target`, as `logQ` **three days in the future**. This makes the
task a genuine three-day-ahead forecast, using only information available today.

7. Drop rows containing NaN (the lags and rolling windows leave gaps at the start of each
catchment's record, and the target leaves gaps at the end). Assemble the feature matrix `X`
from all the engineered features plus `area_sq_mi`, `lat` and `lon`, and the target vector
`y`. Report the shapes and the number of features.

## Part 3: Splitting and scaling

8. Split the data **chronologically**, not randomly: train on everything before 2014,
validate on 2014–2016, and test on 2017 onwards. Report the size of each split.

In a comment, explain why a random split would give a dishonestly good score here. Refer to
what you learned in Assignment 5.

9. Scale the features with `StandardScaler`. Fit the scaler on the **training set only**,
then transform all three sets with it. Explain in a comment what would leak if you fitted it
on everything.

## Part 4: Train a neural network

10. Using `tensorflow`, build a fully connected network that takes the feature matrix as
input and has three hidden `Dense` layers of 128, 64 and 32 units with `ReLU` activation,
followed by a final `Dense` layer with a single output and no activation.

11. Print the model summary. How many trainable parameters does it have?

12. Compile the model with MSE loss and the `Adam` optimizer, including root mean squared error as a metric.

13. Train the model for up to 200 epochs using the validation set, with an `EarlyStopping`
callback (`patience=20`, `restore_best_weights=True`). Report the epoch it stopped at.

14. Plot the training and validation loss against epoch number.

15. Save the trained model.

## Part 5: Evaluate honestly

16. Evaluate the model on the **test** set and report the MSE and RMSE. Then compute the
$R^2$ between predicted and observed `logQ`.

17. A score means nothing without a baseline. Compare your network against two:

- **Persistence**: predict that the flow in three days equals today's flow
  (`logQ_lag0`). This is what a forecaster with no model at all would say, and on a river
  it is a genuinely strong prediction.
- **Linear regression**: the same features, fitted with `LinearRegression`.

Report $R^2$ for all three on the test set, in one table.

18. Make a scatter plot of predicted against observed `logQ` for the test set, with a 1:1 line. Where in the range does the model do worst?

19. Plot the observed and predicted hydrograph for one catchment over a single test year, so
that the two curves can be compared day by day. Does the model capture the timing of the
peaks? Does it capture their magnitude?

## Part 6: Interpretation

20. You trained a single network on six catchments at once, rather than six separate
networks. Explain why pooling helps a neural network here, and what the `area_sq_mi`, `lat`
and `lon` features are doing in a pooled model.

*Write your answer here.*

21. Compare the error on the highest flows against the error on typical flows. Floods are
the reason anyone builds this model, and they are the hardest part of the range to get
right. Give two reasons why, and suggest one concrete change to the training procedure that
would prioritize them.

*Write your answer here.*

22. This model forecasts three days ahead using *observed* precipitation up to today. A real
operational flood forecast needs to run further ahead than that. What would have to change,
and what new source of error would that introduce?

*Write your answer here.*